# 🏆 The ML Gold Standard: Business-First Engineering Manifesto
**Version 3.0 — Focus: Categorical Precision & Explainable AI**
**Effective Date:** March 8, 2026

This notebook serves as the technical implementation of my professional standards for **Supervised Classification**. It is designed to transform raw data into high-confidence decision engines.

> **Core Philosophy:** I solve real-world business problems by applying rigorous engineering standards to messy data. I don't just "classify"—I provide the probability and the "why" behind every prediction to ensure business trust.

---

### 🛡️ The Architectural Pillars
This template provides a comprehensive end-to-end pipeline for professional-grade classification:

* **The Sanity Gate:** Automated enforcement that identifies and removes zero-variance features, high-cardinality IDs, and sparse columns before they can infect the model.

* **The Wall of Silence:** Strict separation of training and testing data using **Stratified Splitting** to ensure class distributions are preserved and leakage is mathematically impossible.

* **The Transformation Engine:** A robust, leakage-free pipeline that handles numeric scaling and categorical encoding, treating missingness as a business signal.

* **SOTA Benchmarking:** Comparing "Glass-Box" Logistic Regression against complex ensembles to ensure every bit of model complexity is justified by a >10% performance gain.

* **Scientific Rigor & Bias Audit:** Beyond simple accuracy: evaluating performance via Stratified Cross-Validation and auditing model fairness across specific business segments.

* **Interpretability & Handover:** Mapping the "Top 10 Business Drivers" via feature importance and securing the asset through automated model persistence.

* **The Validation Layer:** A functional, interactive "What-If" dashboard built for stakeholders to test model intuition and prediction confidence in real-time.

---

**Execution Protocol:** Adjust each cell sequentially using LLM assistance while maintaining the underlying engineering principles defined in this manifesto.

In [ ]:
# COMMANDMENT 0: INFRASTRUCTURE SETUP
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import ipywidgets as widgets
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer, IterativeImputer
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay

# Create Manifesto Directory Structure
for folder in ['data', 'visuals', 'Presentation']:
    os.makedirs(folder, exist_ok=True)

print("✅ Directory Structure Created: data/, visuals/, Presentation/")

In [ ]:
# COMMANDMENT 0.5: THE SANITY CHECK (DATA CONTRACT)
def run_sanity_check(df):
    initial_cols = df.columns.tolist()
    # 1. Drop Constant Features (Zero Variance)
    df = df.loc[:, df.nunique() > 1]

    # 2. Flag High Cardinality (Potential IDs) - unique values > 90% of data
    threshold = 0.9
    potential_ids = [col for col in df.columns if (df[col].nunique() / len(df)) > threshold and not np.issubdtype(df[col].dtype, np.number)]
    df = df.drop(columns=potential_ids)

    # 3. Drop columns with > 80% missingness
    df = df.dropna(thresh=len(df) * 0.2, axis=1)

    dropped = set(initial_cols) - set(df.columns)
    print(f"🧹 Sanity Check Complete. Dropped {len(dropped)} columns: {dropped}")
    return df

# Apply before splitting
df = run_sanity_check(df)

In [ ]:
# COMMANDMENT 1: THE WALL OF SILENCE
# Loading a sample dataset for demonstration (Replace with your data/)
from sklearn.datasets import load_breast_cancer
data = load_breast_cancer()
df = pd.DataFrame(data.data, columns=data.feature_names)
df['target'] = data.target

X = df.drop('target', axis=1)
y = df['target']

# Rule: Split before ANY analysis to prevent leakage
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

print(f"✅ Data Split Complete. Train shape: {X_train.shape}, Test shape: {X_test.shape}")

In [ ]:
# COMMANDMENT 2: MISSINGNESS AS A SIGNAL
def add_missing_indicators(X_df):
    X_copy = X_df.copy()
    for col in X_copy.columns:
        if X_copy[col].isnull().sum() > 0:
            X_copy[f'{col}_is_missing'] = X_copy[col].isnull().astype(int)
    return X_copy

X_train = add_missing_indicators(X_train)
X_test = add_missing_indicators(X_test)
print("✅ Missing value signals captured as boolean features.")

In [ ]:
# COMMANDMENT 3: THE TRANSFORMATION ENGINE
numeric_features = X_train.select_dtypes(include=['int64', 'float64']).columns
categorical_features = X_train.select_dtypes(include=['object']).columns

# Standardizing numeric, Imputing categorical
numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numeric_features),
        ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_features)
    ])

print("✅ Pipeline Preprocessor defined. Ready for leakage-free fitting.")

In [ ]:
# COMMANDMENT 4 & 5: BASELINE VS SOTA & KPI SELECTION
SCORING_METRIC = 'f1' # Choosing F1 to balance Precision/Recall

# 1. Simple Baseline (Glass Box)
baseline_model = Pipeline(steps=[('pre', preprocessor), ('m', LogisticRegression())])

# 2. Complex Model (SOTA)
complex_model = Pipeline(steps=[('pre', preprocessor), ('m', RandomForestClassifier(n_estimators=100))])

print(f"✅ Baseline and Complex models ready. Optimizing for: {SCORING_METRIC}")

In [ ]:
# COMMANDMENT 6: STABILITY CHECK
def validate_stability(model, name):
    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    scores = cross_validate(model, X_train, y_train, cv=cv, scoring=SCORING_METRIC)
    mean_s = scores['test_score'].mean()
    std_s = scores['test_score'].std()
    print(f"🚀 {name} {SCORING_METRIC.upper()}: {mean_s:.4f} (+/- {std_s:.4f})")
    return mean_s

b_score = validate_stability(baseline_model, "Baseline")
c_score = validate_stability(complex_model, "Complex")

# Rule 4 Check: Is Complex >5% better than Baseline?
improvement = (c_score - b_score) / b_score
print(f"📈 Relative Improvement: {improvement:.2%}")

In [ ]:
# COMMANDMENT 7 & 8: ERROR ANALYSIS & RIGOR
complex_model.fit(X_train, y_train)
y_pred = complex_model.predict(X_test)

print(classification_report(y_test, y_pred))
ConfusionMatrixDisplay.from_predictions(y_test, y_pred, cmap='Greens')
plt.title("Post-Mortem Error Analysis")
plt.savefig('visuals/error_analysis.png')
plt.show()

In [ ]:
# COMMANDMENT 8.1: THE BIAS AUDIT (PERFORMANCE SLICING)
from sklearn.metrics import f1_score

slice_feature = X_test.columns[0] # Adjust to a segment like 'Department' or 'Region'
test_df = X_test.copy()
test_df['actual'] = y_test
test_df['pred'] = complex_model.predict(X_test)

print(f"📊 Performance Audit by Segment: {slice_feature}")
for segment in test_df[slice_feature].unique()[:3]:
    subset = test_df[test_df[slice_feature] == segment]
    score = f1_score(subset['actual'], subset['pred'], average='weighted')
    print(f"   - Segment '{segment}': F1-Score = {score:.4f}")

In [ ]:
# COMMANDMENT 9: FEATURE IMPORTANCE (INTERPRETABILITY)
# We pull importance from the Complex Model (Random Forest)
final_model = complex_model.named_steps['m']

# Get feature names after preprocessing
# Note: This assumes you have 'numeric_features' and 'categorical_features' defined
ohe_features = complex_model.named_steps['pre'].transformers_[1][1].get_feature_names_out(categorical_features)
all_features = np.concatenate([numeric_features, ohe_features])

# Create Importance DataFrame
importances = pd.DataFrame({
    'feature': all_features,
    'importance': final_model.feature_importances_
}).sort_values('importance', ascending=False).head(10)

# Visualization
plt.figure(figsize=(10, 6))
sns.barplot(x='importance', y='feature', data=importances, palette='viridis')
plt.title("Top 10 Drivers of Classification")
plt.xlabel("Gini Importance Score")
plt.ylabel("Feature")
plt.tight_layout()
plt.savefig('visuals/classification_feature_importance.png')
plt.show()

print("✅ Feature importance mapped. Model is now a 'Glass Box'.")

In [ ]:
# COMMANDMENT 9.5: THE HANDOVER (MODEL PERSISTENCE)
import joblib
import datetime

timestamp = datetime.datetime.now().strftime("%Y%m%d")
model_path = f'Presentation/classification_v3_{timestamp}.pkl'

# Save the entire pipeline (Preprocessor + Model)
joblib.dump(complex_model, model_path)

print(f"📦 Model Persistence Secured: {model_path}")

In [ ]:
# COMMANDMENT 10: VALIDATION LAYER (STAKEHOLDER DASHBOARD)
def quick_predict(**kwargs):
    row = pd.DataFrame([kwargs])
    for col in X_train.columns:
        if col not in row.columns: row[col] = 0

    # Ensure column order matches training
    row = row[X_train.columns]

    pred = complex_model.predict(row)[0]
    conf = complex_model.predict_proba(row).max()

    result = "POSITIVE" if pred == 1 else "NEGATIVE"
    print(f"--- Prediction: {result} ({conf:.2%} Confidence) ---")

# IMPROVEMENT: Automatically pull the Top 5 features from your Importance DataFrame
top_features = importances['feature'].values[:5]

ui_elements = {f: widgets.FloatSlider(
    min=float(X_train[f].min()),
    max=float(X_train[f].max()),
    value=float(X_train[f].mean()),
    description=f[:15]
) for f in top_features}

print("🎮 INTERACTIVE CLASSIFICATION DASHBOARD (TOP DRIVERS)")
widgets.interact(quick_predict, **ui_elements);

In [ ]:
# COMMANDMENT 11: SELECTIVE ENVIRONMENT LOCK (FOR AWS)
import subprocess

req_path = 'Presentation/requirements.txt'
# We only lock the engines actually used in the Gold Standard
core_libraries = ['pandas', 'numpy', 'matplotlib', 'seaborn', 'scikit-learn', 'xgboost', 'joblib', 'ipywidgets']

# Get all installed packages from Colab
all_packages = subprocess.check_output(['pip', 'freeze']).decode('utf-8').split('\n')

# Filter for a lean production environment
with open(req_path, 'w') as f:
    for pkg in all_packages:
        if any(lib in pkg.lower() for lib in core_libraries):
            f.write(pkg + '\n')

print(f"✅ Selective Environment Locked: {req_path}")
print("📦 Upload this file to AWS to ensure your pipeline matches your Colab experiment.")